# Querying data.gov.cz
* trying GraphQL and SPARQL

## Using graphql
GraphQL is not as powerful as SPARQL.

In [1]:
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

transport = RequestsHTTPTransport(
    url="https://data.gov.cz/graphql",
    verify=True,
    retries=3,
)

client = Client(transport=transport, fetch_schema_from_transport=False)

query = gql("""
{
  datasets(limit: 10) {
    data {
      iri
      title {
        cs
      }
      description {
        cs
      }
      publisher {
        title {
          cs
        }
      }
    }
  }
}
""")
result = client.execute(query)
result

{'datasets': {'data': [{'iri': 'https://data.gov.cz/zdroj/datové-sady/48135097/e9b7837b0f81d4185c2b3ff769c2bc7b',
    'title': {'cs': 'Patenty a užitné vzory - přírůstek 2023-11-17'},
    'description': {'cs': 'Datová sada obsahuje data zveřejněných přihlášek vynálezů, udělených patentů a zapsaných užitných vzorů. Data jsou publikovaná ve formě ročního plného exportu a denních přírůstkových aktualizací.'},
    'publisher': {'title': {'cs': 'Úřad průmyslového vlastnictví'}}},
   {'iri': 'https://data.gov.cz/zdroj/datové-sady/48135097/4423e3ec5b66e88c39edc26ba4cf363b',
    'title': {'cs': 'Patenty a užitné vzory - plný export 2023-01-16 část 55'},
    'description': {'cs': 'Datová sada obsahuje data zveřejněných přihlášek vynálezů, udělených patentů a zapsaných užitných vzorů. Data jsou publikovaná ve formě ročního plného exportu a denních přírůstkových aktualizací.'},
    'publisher': {'title': {'cs': 'Úřad průmyslového vlastnictví'}}},
   {'iri': 'https://data.gov.cz/zdroj/datové-sady/

## Using SPARQL
SPARQL is more complex, but more powerful.

In [2]:
from rdflib import Graph
from rdflib.plugins.stores.sparqlstore import SPARQLStore

# Connect to NKOD SPARQL endpoint
endpoint = "https://data.gov.cz/sparql"
store = SPARQLStore(endpoint)
g = Graph(store)

# Define your query
query = """
PREFIX ds: <https://data.gov.cz/zdroj/datové-sady/00025593/>

SELECT DISTINCT ?property ?value
WHERE {
    ds:719f7b9f2cf4ab7fa40c7e7c459995a7 ?property ?value .
}
LIMIT 100
"""

for prop, val in g.query(query):
    print(f"{prop} -> {val}")

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#duration, Converter=<function parse_xsd_duration at 0x7fe491a12de0>
Traceback (most recent call last):
  File "/home/lucka/prog/EnhancedDatasetSearch/venv/lib/python3.12/site-packages/rdflib/term.py", line 2163, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
           ^^^^^^^^^^^^^^^^^^
  File "/home/lucka/prog/EnhancedDatasetSearch/venv/lib/python3.12/site-packages/rdflib/xsd_datetime.py", line 433, in parse_xsd_duration
    raise ValueError("Unable to parse duration string " + dur_string)
ValueError: Unable to parse duration string 0


http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> https://data.gov.cz/slovník/nkod/typ-datové-sady-dle-zdroje/DokumentyDCAT
http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> https://data.gov.cz/slovník/nkod/typ-datové-sady-dle-zdroje/LKOD
http://www.w3.org/1999/02/22-rdf-syntax-ns#type -> http://www.w3.org/ns/dcat#Dataset
http://purl.org/dc/terms/title -> Cizinci podle státního občanství, věku a pohlaví - rok 2018
http://purl.org/dc/terms/description -> Datová sada obsahuje statistické údaje o počtech cizinců (bez azylantů, s pobytem přechodným, dlouhodobým, pobytem trvalým a dlouhodobým vízem) v České republice, a v členění podle pohlaví, pětiletých věkových skupin a státního občanství cizince. Údaje jsou k dispozici za okresy České republiky. Připraveny jsou rovněž úhrny za Českou republiku, všechna státní občanství, pohlaví i věkové skupiny.
http://purl.org/dc/terms/identifier -> https://vdb.czso.cz/pll/eweb/lkod_ld.datova_sada?nazev=Cizinci_podle_statniho_obcanstvi_veku_a_pohlavi

In [1]:
from rdflib import Graph
from rdflib.plugins.stores.sparqlstore import SPARQLStore

endpoint = "https://data.gov.cz/sparql"
g = Graph(SPARQLStore(endpoint))

query = """
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX dcat: <http://www.w3.org/ns/dcat#>
PREFIX schema: <http://schema.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?dataset (?start AS ?startStr) (?end AS ?endStr) ?spatialName
WHERE {
    ?dataset dct:temporal ?period ;
             dct:spatial ?spatial .

    OPTIONAL { ?period dcat:startDate ?start }
    OPTIONAL { ?period dcat:endDate ?end }

    OPTIONAL { ?spatial schema:name ?schemaName }
    OPTIONAL { ?spatial rdfs:label ?rdfsLabel }
    BIND(COALESCE(?schemaName, ?rdfsLabel) AS ?spatialName)

    FILTER(!BOUND(?spatialName) || lang(?spatialName) = "cs")
}
"""

for dataset, start, end, spatial in g.query(query):
    if spatial and str(spatial) != "Česká republika":
        print("Dataset:", str(dataset))
        print("Start: ", str(start))
        print("End:", str(end))
        print("Region:", str(spatial))
        print("-" * 60)

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x7ff9551c8900>
Traceback (most recent call last):
  File "/home/lucka/prog/EnhancedDatasetSearch/venv/lib/python3.12/site-packages/rdflib/term.py", line 2163, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
           ^^^^^^^^^^^^^^^^^^
  File "/home/lucka/prog/EnhancedDatasetSearch/venv/lib/python3.12/site-packages/rdflib/xsd_datetime.py", line 593, in parse_xsd_date
    return parse_date(date_string if not minus else ("-" + date_string))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: Invalid isoformat string: '2010-01'
Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#date, Converter=<function parse_xsd_date at 0x7ff9551c8900>
Traceback (most recent call last):
  File "/home/lucka/prog/EnhancedDatasetSearch/venv/lib/python3.12/site-packages/

Dataset: https://data.gov.cz/zdroj/datové-sady/00845451/43b10c439031bac33f1f5df1d268e121
Start:  2008-01-01
End: 2008-12-31
Region: Ostrava
------------------------------------------------------------
Dataset: https://data.gov.cz/zdroj/datové-sady/00845451/7a0cf2f18c0a1e35bf0002d5f42ba004
Start:  2022-01-01
End: 2022-12-31
Region: Ostrava
------------------------------------------------------------
Dataset: https://data.gov.cz/zdroj/datové-sady/70892156/6a4915143404869622099dc01c11e5f4
Start:  2025-06-25
End: 2025-06-25
Region: Ústecký kraj
------------------------------------------------------------
Dataset: https://data.gov.cz/zdroj/datové-sady/70892156/6a15527ef9a1e400caad9f3c8b41c0d4
Start:  2021-01-01
End: None
Region: Ústecký kraj
------------------------------------------------------------
Dataset: https://data.gov.cz/zdroj/datové-sady/70892156/838b9fdcbd0c20e30c6c77b16f011cfb
Start:  2024-01-01
End: None
Region: Ústecký kraj
-----------------------------------------------------

In [4]:
query = """
PREFIX dct: <http://purl.org/dc/terms/>
PREFIX ds: <https://data.gov.cz/zdroj/datové-sady/00025593/>
PREFIX lsgov: <https://slovník.gov.cz/legislativní/sbírka/111/2009/pojem/>

SELECT ?poskytovatel ?nazev
WHERE {
  ?poskytovatel lsgov:má-název-orgánu-veřejné-moci ?nazev .
}
LIMIT 10
"""

for owner, title, in g.query(query):
    print("Poskytovatel:", str(owner))
    print("Název:", str(title))
    print("-" * 60)

Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/00852546
Název: Základní škola Krnov, Janáčkovo náměstí 17, okres Bruntál, příspěvková organizace
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/06959920
Název: Mateřská škola Chlístovice, příspěvková organizace
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/17490863
Název: Česká lékařská komora se sídlem v Olomouci, okresní sdružení Plzeň-sever
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/00854727
Název: Waldorfská mateřská škola Semily, příspěvková organizace
------------------------------------------------------------
Poskytovatel: https://rpp-opendata.egon.gov.cz/odrpp/zdroj/orgán-veřejné-moci/08388962
Název: Mateřská škola Polerady